In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from IPython.display import display


url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(url)

df. columns = df.columns.str.lower().str.replace(" ", "_")
str_cols = list(df.dtypes[df.dtypes == 'str'].index)

for c in str_cols:
    df[c] =df[c].str.lower().str.replace(" ", "_")

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn=='yes').astype('int')

categorical = [
    'gender', 'seniorcitizen', 'partner', 'dependents',
    'phoneservice', 'multiplelines', 'internetservice',
    'onlinesecurity', 'onlinebackup', 'deviceprotection',
    'techsupport', 'streamingtv', 'streamingmovies',
    'contract', 'paperlessbilling', 'paymentmethod'
]


In [40]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop = True)
df_val = df_val.reset_index(drop = True)
df_test = df_test.reset_index(drop = True)

y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

del df_train['churn']
del df_val['churn']
del df_test['churn']

df_full_train.columns


Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='str')

In [52]:
df_full_train = df_full_train.reset_index(drop=True)
df_full_train.isnull().sum()
df_full_train.churn.value_counts(normalize=True)

global_churn = df_full_train.churn.values.mean()

# for c in cats_cols:
#     print(c)
#     df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
#     df_group['diff'] = df_group['mean'] - global_churn
#     df_group['risk'] = df_group['mean'] / global_churn
#     display(df_group)
#     print()
#     print()

In [55]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

numerical = ["tenure", "monthlycharges", "totalcharges"]
df_full_train[numerical].corrwith(df_full_train.churn)

tenure           -0.351885
monthlycharges    0.196805
totalcharges     -0.196353
dtype: float64

In [81]:
train_dict = df_train[numerical+categorical].to_dict(orient="records")
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)

val_dict = df_val[numerical + categorical].to_dict(orient="records")
X_val = dv.transform(val_dict)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))
model = LogisticRegression(solver='liblinear', max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]
churn_decision = (y_pred >= 0.51)



In [85]:
full_train_dict = df_full_train[numerical + categorical].to_dict(orient = "records")
dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(full_train_dict)
y_full_train = df_full_train.churn.values

model = LogisticRegression(solver="liblinear", max_iter=1000)
model.fit(X_full_train, y_full_train)

test_dict = df_test[numerical+categorical].to_dict(orient="records")
X_test = dv.transform(test_dict)
y_pred = model.predict_proba(X_test)[:, 1]
churn_decision = (y_pred >= 0.5)
(churn_decision == y_test).mean()

np.float64(0.8126330731014905)